In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "albiach2014reversed")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "22 05 05_Albiach-Serrano & Call, 2014_Experiment1.sav")
complete_path_2 = os.path.join(original_data_pathway, "22 05 05_Albiach-Serrano & Call, 2014_Experiment2.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat


df1 = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
df1 = df1.assign(experiment='1')
df1.rename(columns={"Age": "age_in_years"}, inplace=True)

In [3]:

df2 = pd.read_spss(complete_path_2, usecols=None, convert_categoricals=True)
df2 = df2.assign(experiment='2')

# df2.columns
df2.rename(columns={"Age_years": "age_in_years"}, inplace=True)

In [4]:
data_frames=[df1, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"subject": "ape",
        'sex':'sex_original',
        'continuousstrip':'continuous_strip'}, inplace=True)
    x['ape'] = x['ape'].str.rstrip()
    x['study_id']="albiach2014reversed"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

In [5]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['ape'] = fulldf['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')

In [6]:
fulldf.replace(' ', '_', inplace=True, regex=True)

fulldf.replace('\(', '', inplace=True, regex=True)
fulldf.replace('\)', '', inplace=True, regex=True)

In [7]:
# fulldf.columns
fulldf.rename(columns={"ape": "participant",
                       "age_months":"age_original"}, inplace=True)

In [8]:
fulldf = fulldf[['study_id', 'experiment',
        'participant', 'age_original','age_in_years',  'sex', 'species', 'session', 'trial', 'contingency', 'condition', 
       'order',  'pattern', 'bait', 
        'continuous_strip' ,'choice', 'correct' ]]

In [9]:
for index in range(1,3):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'albiach2014reversed_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'albiach2014reversed_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)